# **Time Series Analysis 2026 — Capstone Project**

**Student Name**: Abhishek Raghuwanshi

**Registered Phone Number**: 8839661736

**Email**: 9905abhishek@gmail.com

## Project Overview
This project applies time series forecasting to predict NSE stock prices and construct a ₹10,00,000 virtual portfolio on StockGro.
The pipeline includes:
1. Stock Universe Selection (Task 1)
2. Data Preprocessing (Task 2)
3. Time Series Forecasting (Task 3)
4. Volatility & Trend Analysis (Task 4)
5. Portfolio Construction (Task 5)
6. Model Comparison (Task 6)
7. Virtual Trading on StockGro (Task 7)
8. Predicted vs Actual Comparison (Task 8)

## Key Libraries Used
- `yfinance`: Downloading historical stock data
- `pandas`, `numpy`: Data manipulation
- `matplotlib`, `seaborn`, `plotly`: Visualization
- `pmdarima`: Auto ARIMA modeling
- `prophet`: Facebook Prophet forecasting
- `torch`: PyTorch for LSTM sequence modeling
- `statsmodels`: Exponential smoothing, STL decomposition, ADF tests
- `arch`: GARCH volatility modeling

### **Import Libraries**

In [ ]:
# Import required libraries
import sys, os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Custom project modules
sys.path.insert(0, '.')
from src.data_fetcher import fetch_all_stocks, SECTOR_MAP, get_all_tickers
from src.preprocessor import preprocess_stock
from src.evaluation import compute_mape, compute_rmse

%matplotlib inline
plt.style.use('dark_background')
print('All libraries imported successfully.')

## **Time Series Data**
### Loading data & Stock Selection (Task 1)

In [ ]:
# Retrieve historical stock price data (Jan 2021 - Dec 2025)
tickers = get_all_tickers()
raw_data = fetch_all_stocks(tickers, cache_dir='data/raw')

print(f'\nSelected Stock Universe ({len(tickers)} stocks):')
for sector, tkrs in SECTOR_MAP.items():
    names = ', '.join(t.replace('.NS', '') for t in tkrs)
    print(f'  {sector:>10}: {names}')

### Plotting historical stock data

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
colors = ['#00D4AA','#FF6B6B','#4ECDC4','#FFE66D','#95E1D3',
          '#A8E6CF','#FF8B94','#F38181','#AA96DA','#FCBAD3','#6C5B7B']
for i, (ticker, df) in enumerate(raw_data.items()):
    ax.plot(df.index, df['Close'], label=ticker.replace('.NS',''), color=colors[i % len(colors)], linewidth=1.2)
ax.set_title('Stock Universe - Daily Close Prices (2021-2025)', color='#c9d1d9')
ax.set_ylabel('Price (INR)', color='#8b949e')
ax.legend(ncol=3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### Creating train and test datasets & Scaling (Task 2)
Preprocessing includes handling missing values, ADF stationarity tests, differencing, min-max scaling for LSTM, and splitting.

In [ ]:
processed = {}
print(f'{"Stock":>15} {"ADF p-value":>12} {"Stationary?":>12} {"Train":>7} {"Test":>6}')
print('-' * 60)
for ticker, df in raw_data.items():
    processed[ticker] = preprocess_stock(df)
    name = ticker.replace('.NS', '')
    adf = processed[ticker]['adf_result']
    n_train, n_test = len(processed[ticker]['train']), len(processed[ticker]['test'])
    stat = 'Yes' if adf['is_stationary'] else 'No'
    print(f'{name:>15} {adf["p_value"]:>12.4f} {stat:>12} {n_train:>7} {n_test:>6}')

## **Time Series Prediction Models** (Task 3 & 4)
Models evaluated: ARIMA, Prophet, LSTM, Exponential Smoothing. GARCH used for volatility.

In [ ]:
# Load pre-computed forecasts and GARCH volatility results
with open('outputs/forecasts/all_forecasts.json') as f:
    all_forecasts = json.load(f)
with open('outputs/task4_garch.json') as f:
    garch_results = json.load(f)

print(f'{"Stock":>13} {"ARIMA D1":>10} {"ARIMA D2":>10} | {"GARCH Vol":>10}')
print('-' * 55)
for ticker, models in all_forecasts.items():
    fc = models['ARIMA']['forecast']
    vol = garch_results[ticker]['avg_daily_vol']
    name = ticker.replace('.NS', '')
    print(f'{name:>13} {fc[0]:>10.2f} {fc[1]:>10.2f} | {vol:>9.4f}%')

### Time Series Prediction: Actual vs Predicted on Test Set

In [ ]:
example_ticker = 'HDFCBANK.NS'
test_actual = processed[example_ticker]['test']['Close'].values
test_pred = all_forecasts[example_ticker]['ARIMA']['test_predictions']

plt.figure(figsize=(14, 5), facecolor='#0d1117')
ax = plt.gca()
ax.set_facecolor('#161b22')
ax.plot(test_actual, label='Actual', color='#00D4AA')
ax.plot(test_pred, label='ARIMA Predicted', color='#FF6B6B', linestyle='--')
ax.set_title('HDFC Bank - ARIMA Test Set Predictions', color='#c9d1d9')
ax.legend()
plt.show()

## **Portfolio Construction** (Task 5)
Allocating ₹10,00,000 using Forecast-Guided, Volatility-Aware, Correlation-Diversified, and Sector Momentum strategies.

In [ ]:
allocation = pd.read_csv('outputs/task5_allocation.csv')
print(allocation.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 8), facecolor='#0d1117')
ax.pie(allocation.iloc[:,1].values, labels=[str(s).replace('.NS','') for s in allocation.iloc[:,0]], autopct='%1.1f%%', textprops={'color': '#c9d1d9'})
ax.set_title('Final Portfolio Allocation', color='#c9d1d9')
plt.show()

## **Model Comparison** (Task 6)

In [ ]:
comparison = pd.read_csv('outputs/task6_comparison.csv')
print(comparison.to_string(index=False))
print('\nARIMA selected for portfolio allocation due to best (lowest) MAPE.')

## **StockGro Virtual Trading & Results** (Task 7 & 8)

In [ ]:
with open('outputs/task8_actuals.json') as f:
    actuals = json.load(f)
with open('outputs/forecasts/all_forecasts.json') as f:
    forecasts = json.load(f)

rows = []
for ticker, info in actuals['stocks'].items():
    pred = forecasts[ticker]['ARIMA']['forecast']
    act_d1, act_d2 = info['day1_ltp'], info['day2_ltp']
    err_d2 = abs(pred[1] - act_d2) / act_d2 * 100
    rows.append({'Stock': ticker.replace('.NS',''), 'Pred D2': round(pred[1],2), 'Actual D2': act_d2, 'Error %': round(err_d2,2)})

print(pd.DataFrame(rows).to_string(index=False))
print(f'\nInvested: Rs.{actuals["invested_value"]:,.0f}')
print(f'Day 2 Value: Rs.{actuals["day2_portfolio_value"]:,.0f} ({actuals["day2_return_pct"]:+.2f}%)')
print(f'P&L: Rs.{actuals["day2_total_loss"]:,.0f}')

## **Conclusion**
- **What Worked**: ARIMA model performed excellently in backtesting (0.77% MAPE). GARCH effectively captured volatility signatures.
- **What Didn't Work**: Gap between Dec 2025 training end and May 2026 trading window caused significant live prediction errors (avg 20% MAPE) due to market shifts.
- **Improvements**: Retrain models daily on the most recent data, add exogenous macro-economic variables, and implement stop-loss strategies.